# Course 6: Winning Online AI Competitions
## Session 2: Metrics and leaderboards

**Goal:** compute AUC, log loss and F1 by hand, see what each one rewards, and simulate a public and private leaderboard 200 times.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. Three metrics by hand

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss, f1_score
rng = np.random.default_rng(0)
y = (rng.random(2000) < 0.2).astype(int)
p = np.clip(0.2 + 0.25 * (y - 0.2) + rng.normal(scale=0.15, size=2000), 0.01, 0.99)       # a decent model's probabilities

def auc(y, p):
    order = np.argsort(p); ranks = np.empty(len(p)); ranks[order] = np.arange(1, len(p) + 1)
    n1, n0 = y.sum(), (1 - y).sum()
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)             # chance that a random positive outranks a random negative
def logloss(y, p): return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
def f1(y, yhat):
    tp = ((yhat == 1) & (y == 1)).sum(); fp = ((yhat == 1) & (y == 0)).sum(); fn = ((yhat == 0) & (y == 1)).sum()
    return 2 * tp / (2 * tp + fp + fn)
print("AUC      %.4f  (sklearn %.4f)" % (auc(y, p), roc_auc_score(y, p)))
print("log loss %.4f  (sklearn %.4f)" % (logloss(y, p), log_loss(y, p)))
print("F1 at 0.5 %.4f (sklearn %.4f)" % (f1(y, (p > 0.5).astype(int)), f1_score(y, p > 0.5)))
assert abs(auc(y, p) - roc_auc_score(y, p)) < 1e-9 and abs(logloss(y, p) - log_loss(y, p)) < 1e-9

### 2. What each metric rewards
- AUC looks only at the order of the predictions. Any change that keeps the order leaves it unchanged.
- Log loss needs the probabilities themselves to be right. One confident mistake is very expensive.
- F1 needs a yes/no decision, so the threshold is part of the model.

In [ ]:
squashed = p ** 3                                                    # same order, very different probabilities
print("squashing the probabilities:  AUC %.4f -> %.4f   log loss %.4f -> %.4f" % (auc(y, p), auc(y, squashed), logloss(y, p), logloss(y, squashed)))
p_bad = p.copy(); p_bad[np.flatnonzero(y == 1)[:5]] = 1e-6            # five confident mistakes out of 2,000 rows
print("five confident mistakes:      AUC %.4f -> %.4f   log loss %.4f -> %.4f" % (auc(y, p), auc(y, p_bad), logloss(y, p), logloss(y, p_bad)))
ths = np.linspace(0.05, 0.95, 91); f1s = [f1(y, (p > t).astype(int)) for t in ths]
print("F1 at threshold 0.5: %.3f | best F1 %.3f at threshold %.2f" % (f1(y, (p > 0.5).astype(int)), max(f1s), ths[int(np.argmax(f1s))]))
assert abs(auc(y, p) - auc(y, squashed)) < 1e-12 and logloss(y, p_bad) > logloss(y, p) + 0.01

So: for AUC, rank-average your models and do not worry about calibration. For log loss, clip predictions away from 0 and 1 and calibrate. For F1, tune the threshold on out-of-fold predictions, never on the leaderboard.

### 3. Public and private leaderboards
The test set is split in two. You see your score on the public part during the competition. The final ranking uses the private part. Simulate a competition: 200 teams of nearly equal skill, a public part of 20%, and repeat it 200 times.

In [ ]:
def simulate(n_test=2000, public_share=0.2, n_teams=200, n_sims=200, skill_spread=0.004, seed=0):
    rng = np.random.default_rng(seed); n_pub = int(n_test * public_share); shifts, winner_private_rank = [], []
    for _ in range(n_sims):
        true_acc = 0.80 + rng.normal(scale=skill_spread, size=n_teams)               # the teams' real quality
        correct = rng.random((n_teams, n_test)) < true_acc[:, None]
        pub, priv = correct[:, :n_pub].mean(1), correct[:, n_pub:].mean(1)
        r_pub, r_priv = (-pub).argsort().argsort(), (-priv).argsort().argsort()
        shifts.append(np.abs(r_pub - r_priv).mean()); winner_private_rank.append(r_priv[r_pub.argmin()] + 1)
    return np.mean(shifts), np.median(winner_private_rank), np.mean(np.array(winner_private_rank) <= 10)

for share, n_test in [(0.2, 2000), (0.2, 20000), (0.5, 20000)]:
    s, med, top10 = simulate(n_test=n_test, public_share=share)
    print(f"test rows {n_test:6d}, public {int(share * 100)}%: mean rank change {s:5.1f} places | public leader's median private rank {med:4.0f} | stays in the top 10: {top10:.0%}")

With a small public set and closely matched teams, the public leader is mostly lucky. A larger test set helps. Before trusting a public rank, estimate the noise: public rows, the metric, and how close the scores are.

### 4. Overfitting the leaderboard yourself
Now one team makes 100 submissions that are all equally good and keeps the one with the best public score.

In [ ]:
rng = np.random.default_rng(1); n_test, n_pub = 2000, 400
gaps = []
for _ in range(200):
    correct = rng.random((100, n_test)) < 0.80                       # 100 submissions, identical true quality
    pub, priv = correct[:, :n_pub].mean(1), correct[:, n_pub:].mean(1)
    best = pub.argmax(); gaps.append(pub[best] - priv[best])
print(f"best-of-100 by public score: public looks better than private by {np.mean(gaps):.3f} on average (the true quality never changed)")
assert np.mean(gaps) > 0.02

That gap is pure selection. It is why you choose final submissions by cross-validation, and why teams that probe the leaderboard fall on the last day.

### TODO
1. For the competition you joined: how many rows are in the public part? Estimate the standard error of your public score.
2. Add a metric to section 1: mean average precision at 3, or quadratic weighted kappa. Check it against a library.
3. In section 3, give 5 teams a real advantage of 0.02. How often do they finish in the top 5 privately? Publicly?